Data Cleaning and Model Summary

The numerical features such as hours studied, sleep hours, previous scores, and exam score appeared to have approximately normal distributions, although some variables showed slight skewness. Categorical values such as gender, school type, and internet access were not evenly distributed, indicating potential class imbalance in some categories.

Missing values in the dataset were handled by replacing numerical values with the mean of their respective columns and categorical values with the mode. This method was chosen because it preserves the size of the dataset while minimizing the impact of missing data, as opposed to dropping rows which could result in loss of important information.

Categorical variables were encoded using one-hot encoding. This approach was selected because most categorical features in the dataset are nominal and do not have an inherent order. One-hot encoding prevents the model from incorrectly assuming ordinal relationships between categories, which could occur if label encoding were used.

Outliers were identified and removed using the IQR method. This method was chosen because it is robust and does not assume a normal distribution. Removing extreme values helps improve model performance by reducing the influence of anomalous data points.

The data was standardized using StandardScaler, which transforms the features to have a mean of 0 and a standard deviation of 1. This was particularly important for models such as K-Nearest Neighbors, which rely on distance calculations and are sensitive to differences in feature scale.

Four machine learning models were trained and evaluated: linear regression, decision tree, random forest, and k-nearest neighbors. Among these, the linear regression model performed the best, achieving the highest R2 value (0.731) and the lowest RMSE and MAE. Random forest showed moderate performance, while KNN performed less effectively. The decision tree model performed poorly with an R2 value close to zero.

The linear regression model demonstrated a good balance between bias and variance, suggesting that the relationships within the dataset are largely linear. The decision tree model exhibited high variance and likely overfit the training data, resulting in poor generalization to the test set. The random forest model reduced overfitting compared to the decision tree but did not outperform linear regression. The KNN model showed moderate performance and sensitivity to the choice of neighbors.

Overall, the results indicate that the relationship between student characteristics and exam performance is relatively linear. This explains why a simpler model such as linear regression outperformed more complex models. This also highlights that increasing model complexity does not always lead to better predictive performance. 

In [1]:
import pandas as pd

df = pd.read_csv("StudentPerformanceFactors.csv")

In [2]:
df.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


Handle missing values

In [3]:
row_count = len(df)
row_count

6607

In [4]:
df.isna().sum()

Hours_Studied                  0
Attendance                     0
Parental_Involvement           0
Access_to_Resources            0
Extracurricular_Activities     0
Sleep_Hours                    0
Previous_Scores                0
Motivation_Level               0
Internet_Access                0
Tutoring_Sessions              0
Family_Income                  0
Teacher_Quality               78
School_Type                    0
Peer_Influence                 0
Physical_Activity              0
Learning_Disabilities          0
Parental_Education_Level      90
Distance_from_Home            67
Gender                         0
Exam_Score                     0
dtype: int64

In [17]:
# Numerical - mean
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

# Categorical - mode
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

Encode categorical categories 

In [7]:
df = pd.get_dummies(df, drop_first=True)

Identify outliers (focusing on Hours_Studied, Sleep_Hours, Exam_Score)

In [8]:
for col in ['Hours_Studied', 'Sleep_Hours', 'Exam_Score']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 -1

    df = df[(df[col] >= Q1 - 1.5*IQR) &
            (df[col] <= Q3 + 1.5*IQR)]

In [9]:
df.head()

,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions,Physical_Activity,Exam_Score,Parental_Involvement_Low,Parental_Involvement_Medium,Access_to_Resources_Low,...,Teacher_Quality_Medium,School_Type_Public,Peer_Influence_Neutral,Peer_Influence_Positive,Learning_Disabilities_Yes,Parental_Education_Level_High School,Parental_Education_Level_Postgraduate,Distance_from_Home_Moderate,Distance_from_Home_Near,Gender_Male
0,23,84,7,73,0,3,67,True,False,False,...,True,True,False,True,False,True,False,False,True,True
1,19,64,8,59,2,4,61,True,False,False,...,True,True,False,False,False,False,False,True,False,False
2,24,98,7,91,2,4,74,False,True,False,...,True,True,True,False,False,False,True,False,True,True
3,29,89,8,98,1,4,71,True,False,False,...,True,True,False,False,False,True,False,True,False,True
4,19,92,6,65,3,4,70,False,True,False,...,False,True,True,False,False,False,False,False,True,False


Normalize/Standardize data

In [10]:
from sklearn.preprocessing import StandardScaler

x = df.drop('Exam_Score', axis=1)
y = df['Exam_Score']

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

Train the model

In [11]:
from sklearn.model_selection import train_test_split

xTrain, xTest, yTrain, yTest = train_test_split(x_scaled,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=42)

Linear regression model

This model has low variance, and slight bias.

In [12]:
from sklearn.linear_model import LinearRegression
model = LinearRegression().fit(xTrain, yTrain)

In [13]:
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_error

preds = model.predict(xTest)

print(r2_score(yTest, preds))
print(root_mean_squared_error(yTest, preds))
print(mean_absolute_error(yTest, preds))

0.7696017567194431
1.8046317813990218
0.4502854154700954


Decision tree model

This model has high variance (overfitting)

In [14]:
from sklearn import tree

model = tree.DecisionTreeRegressor().fit(xTrain, yTrain)

preds = model.predict(xTest)

print(r2_score(yTest, preds))
print(root_mean_squared_error(yTest, preds))
print(mean_absolute_error(yTest, preds))

-0.0475975200415808
3.848099124032701
1.8895612708018155


Random forest model

This model has reduced variance. It is better than Decision Tree, but not better than Linear Regression.

In [15]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(random_state=42)

model.fit(xTrain, yTrain)
preds = model.predict(xTest)

print(r2_score(yTest, preds))
print(root_mean_squared_error(yTest, preds))
print(mean_absolute_error(yTest, preds))

0.6500691209554543
2.224027503394409
1.1748562783661123


K-nearest neighbors (KNN) model

This model has sensitive/moderate variance.

In [16]:
from sklearn.neighbors import KNeighborsRegressor

model = KNeighborsRegressor(n_neighbors=5)

model.fit(xTrain, yTrain)
preds = model.predict(xTest)

print(r2_score(yTest, preds))
print(root_mean_squared_error(yTest, preds))
print(mean_absolute_error(yTest, preds))

0.42152452674990415
2.8595091778617077
1.906354009077156
